In [ ]:

import os
import time
import random
import copy
import math
from collections import deque, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms

In [ ]:


SEED = 42


def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Seed:", SEED)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Seed: 42
Device: cuda
GPU: Tesla T4


In [ ]:

NUM_CLIENTS = 50

CLIENTS_PER_ROUND = 10

NUM_ROUNDS = 50

NUM_CLASSES = 10


BATCH_SIZE = 64

LEARNING_RATE = 0.001

LOCAL_EPOCH_MIN = 1

LOCAL_EPOCH_MAX = 5


NON_IID_ALPHA = 0.3



RL_GAMMA = 0.95

RL_LR = 1e-3

REPLAY_CAPACITY = 10000

RL_BATCH_SIZE = 64

TARGET_UPDATE_FREQ = 5


EPSILON_START = 1.0

EPSILON_END = 0.05

EPSILON_DECAY = 0.95


CPU_MIN = 1.0
CPU_MAX = 8.0

BATTERY_MIN = 20.0
BATTERY_MAX = 100.0

BANDWIDTH_MIN = 5.0
BANDWIDTH_MAX = 100.0

LATENCY_MIN = 20.0
LATENCY_MAX = 200.0
W_GLOBAL_ACCURACY = 0.45
W_DATA_UTILITY = 0.15
W_TIME = 0.15
W_COMMUNICATION = 0.10
W_ENERGY = 0.10
W_FAIRNESS = 0.05


W_LOCAL_ACCURACY = 0.50
W_LOCAL_TIME = 0.20
W_LOCAL_ENERGY = 0.15
W_LOCAL_COMMUNICATION = 0.15



MODEL_SIZE_MB = 2.0


print("Configuration loaded.")

ENABLE_POST_SELECTION_DROPOUT = True
POST_SELECTION_DROPOUT_PROB = 0.05
ENABLE_MALICIOUS_CLIENT_SIMULATION = True
MALICIOUS_CLIENT_FRACTION = 0.10
MALICIOUS_SIGN_FLIP_STRENGTH = 1.0
DETECTOR_COSINE_THRESHOLD = -0.10
DETECTOR_MAX_NORM_RATIO = 3.0


Configuration loaded.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


classes = train_dataset.classes

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]

    image = image * torch.tensor(
        [0.2470, 0.2435, 0.2616]
    ).view(3, 1, 1)

    image = image + torch.tensor(
        [0.4914, 0.4822, 0.4465]
    ).view(3, 1, 1)

    image = torch.clamp(image, 0, 1)

    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis("off")

plt.suptitle("CIFAR-10 Sample Images", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:

from sklearn.model_selection import train_test_split

all_train_labels = np.asarray(train_dataset.targets)
all_train_indices = np.arange(len(train_dataset))
train_indices, validation_indices = train_test_split(
    all_train_indices, test_size=0.10, random_state=SEED,
    stratify=all_train_labels
)
fl_train_dataset = Subset(train_dataset, train_indices.tolist())
validation_dataset = Subset(train_dataset, validation_indices.tolist())
train_labels = all_train_labels[train_indices]

print(f"FL training samples: {len(fl_train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Test samples (reserved): {len(test_dataset)}")

In [ ]:


def dirichlet_partition(
    labels,
    num_clients,
    num_classes,
    alpha,
    seed
):

    rng = np.random.default_rng(seed)

    client_indices = [
        []
        for _ in range(num_clients)
    ]

    for class_id in range(num_classes):

        class_indices = np.where(
            labels == class_id
        )[0]

        rng.shuffle(class_indices)

        proportions = rng.dirichlet(
            np.repeat(alpha, num_clients)
        )

        split_points = (
            np.cumsum(proportions)
            * len(class_indices)
        ).astype(int)

        split_points[-1] = len(
            class_indices
        )

        splits = np.split(
            class_indices,
            split_points[:-1]
        )

        for client_id, indices in enumerate(splits):

            client_indices[
                client_id
            ].extend(
                indices.tolist()
            )

    for client_id in range(num_clients):

        rng.shuffle(
            client_indices[client_id]
        )

    return client_indices

In [ ]:


client_data = dirichlet_partition(
    labels=train_labels,
    num_clients=NUM_CLIENTS,
    num_classes=NUM_CLASSES,
    alpha=NON_IID_ALPHA,
    seed=SEED
)


print(
    "Created",
    len(client_data),
    "clients."
)

for client_id in range(10):

    print(
        f"Client {client_id:02d}: "
        f"{len(client_data[client_id])} samples"
    )

In [ ]:


client_class_distribution = np.zeros(
    (NUM_CLIENTS, NUM_CLASSES)
)


for client_id in range(NUM_CLIENTS):

    labels = train_labels[
        client_data[client_id]
    ]

    for class_id in range(NUM_CLASSES):

        client_class_distribution[
            client_id,
            class_id
        ] = np.sum(
            labels == class_id
        )


plt.figure(figsize=(14, 10))

sns.heatmap(
    client_class_distribution,
    cmap="viridis",
    xticklabels=classes,
    yticklabels=[
        f"C{i}"
        for i in range(NUM_CLIENTS)
    ]
)

plt.title(
    f"CIFAR-10 Non-IID Distribution "
    f"(Dirichlet α={NON_IID_ALPHA})"
)

plt.xlabel("Class")
plt.ylabel("Client")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 8: CLIENT DATASET
# ============================================================

class ClientDataset(Dataset):

    def __init__(
        self,
        base_dataset,
        indices
    ):

        self.base_dataset = base_dataset
        self.indices = indices


    def __len__(self):

        return len(self.indices)


    def __getitem__(self, index):

        return self.base_dataset[
            self.indices[index]
        ]

In [ ]:

def initialize_client_resources(
    num_clients,
    seed
):

    rng = np.random.default_rng(seed)

    resources = {}

    for client_id in range(num_clients):

        resources[client_id] = {

            "cpu": rng.uniform(
                CPU_MIN,
                CPU_MAX
            ),

            "battery": rng.uniform(
                BATTERY_MIN,
                BATTERY_MAX
            ),

            "bandwidth": rng.uniform(
                BANDWIDTH_MIN,
                BANDWIDTH_MAX
            ),

            "latency": rng.uniform(
                LATENCY_MIN,
                LATENCY_MAX
            ),

            "availability": rng.uniform(
                0.7,
                1.0
            )
        }

    return resources


client_resources = (
    initialize_client_resources(
        NUM_CLIENTS,
        SEED
    )
)


resource_df = pd.DataFrame.from_dict(
    client_resources,
    orient="index"
)

resource_df.index.name = "client"

display(
    resource_df.head()
)

In [ ]:


fig, axes = plt.subplots(
    2,
    2,
    figsize=(15, 9)
)


resource_df["cpu"].plot(
    kind="bar",
    ax=axes[0, 0]
)

axes[0, 0].set_title(
    "CPU Capability"
)


resource_df["battery"].plot(
    kind="bar",
    ax=axes[0, 1]
)

axes[0, 1].set_title(
    "Battery Level"
)


resource_df["bandwidth"].plot(
    kind="bar",
    ax=axes[1, 0]
)

axes[1, 0].set_title(
    "Bandwidth"
)


resource_df["latency"].plot(
    kind="bar",
    ax=axes[1, 1]
)

axes[1, 1].set_title(
    "Communication Latency"
)


plt.tight_layout()

plt.show()

In [ ]:

class CIFAR10CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 8 * 8,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                NUM_CLASSES
            )
        )


    def forward(self, x):

        x = self.features(x)

        return self.classifier(x)

In [ ]:


model = CIFAR10CNN().to(DEVICE)

dummy = torch.randn(
    8, 3, 32, 32
).to(DEVICE)

output = model(dummy)

print("Output shape:", output.shape)

In [ ]:


def get_model_state(model):

    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def load_model_state(
    model,
    state
):

    model.load_state_dict(state)


def fedavg(
    client_states,
    client_sizes
):

    total_samples = sum(
        client_sizes
    )

    weights = [
        size / total_samples
        for size in client_sizes
    ]

    global_state = {}

    for key in client_states[0].keys():

        global_state[key] = sum(
            weights[i]
            * client_states[i][key]
            for i in range(
                len(client_states)
            )
        )

    return global_state

In [ ]:

validation_loader = DataLoader(validation_dataset, batch_size=256, shuffle=False,
                               num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False,
                         num_workers=2, pin_memory=True)

def evaluate_model(model, loader=validation_loader):
    model.eval(); criterion = nn.CrossEntropyLoss()
    total_loss = 0.0; correct = 0; total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            outputs = model(images); loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / max(total, 1), 100.0 * correct / max(total, 1)


In [ ]:


def train_local_model(
    global_state,
    client_id,
    local_epochs,
    client_data,
    train_dataset,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE
):

    model = CIFAR10CNN().to(DEVICE)

    load_model_state(
        model,
        global_state
    )

    model.train()


    dataset = ClientDataset(
        train_dataset,
        client_data[client_id]
    )


    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )


    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    criterion = nn.CrossEntropyLoss()


    total_loss = 0.0

    total_correct = 0

    total_samples = 0


    start_time = time.time()


    for epoch in range(local_epochs):

        model.train()

        for images, labels in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()


            total_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = (
                outputs.argmax(dim=1)
            )

            total_correct += (
                predictions == labels
            ).sum().item()

            total_samples += (
                labels.size(0)
            )


    training_time = (
        time.time() - start_time
    )


    avg_loss = (
        total_loss
        / max(total_samples, 1)
    )


    local_accuracy = (
        total_correct
        / max(total_samples, 1)
    )


    return {
        "state": get_model_state(model),
        "loss": avg_loss,
        "accuracy": local_accuracy,
        "training_time": training_time,
        "samples": len(dataset)
    }

In [ ]:


def normalize_value(
    value,
    minimum,
    maximum
):

    return np.clip(
        (value - minimum)
        / (maximum - minimum + 1e-8),
        0.0,
        1.0
    )


def normalize_improvement(
    improvement,
    scale=0.10
):

    """
    Maps an accuracy improvement of approximately
    [-scale, +scale] into [0, 1].
    """

    return np.clip(
        (
            improvement + scale
        ) / (2 * scale),
        0.0,
        1.0
    )

In [ ]:

def initialize_client_history():

    history = {}

    for client_id in range(NUM_CLIENTS):

        history[client_id] = {

            "last_loss": 2.30,

            "last_accuracy": 0.10,

            "previous_accuracy": 0.10,

            "participation_count": 0,

            "last_epochs": 5,

            "last_training_time": 0.0,

            "last_energy": 0.0,

            "last_communication": 0.0
        }

    return history

In [ ]:


def update_client_resources(
    resources,
    selected_clients
):

    for client_id in range(NUM_CLIENTS):

        r = resources[client_id]


        if client_id in selected_clients:


            r["battery"] -= np.random.uniform(
                0.5,
                2.0
            )



        r["bandwidth"] += np.random.normal(
            0,
            3
        )

        r["latency"] += np.random.normal(
            0,
            5
        )

        r["cpu"] += np.random.normal(
            0,
            0.2
        )

        r["availability"] += np.random.normal(
            0,
            0.02
        )



        r["battery"] = np.clip(
            r["battery"],
            5,
            100
        )

        r["bandwidth"] = np.clip(
            r["bandwidth"],
            BANDWIDTH_MIN,
            BANDWIDTH_MAX
        )

        r["latency"] = np.clip(
            r["latency"],
            LATENCY_MIN,
            LATENCY_MAX
        )

        r["cpu"] = np.clip(
            r["cpu"],
            CPU_MIN,
            CPU_MAX
        )

        r["availability"] = np.clip(
            r["availability"],
            0.5,
            1.0
        )

In [ ]:

def calculate_data_diversity(
    client_id,
    client_data,
    train_labels
):

    labels = train_labels[
        client_data[client_id]
    ]

    unique_classes = len(
        np.unique(labels)
    )

    return (
        unique_classes
        / NUM_CLASSES
    )

In [ ]:


def build_client_state(
    client_id,
    client_data,
    train_labels,
    resources,
    history
):


    data_size = len(
        client_data[client_id]
    )

    max_data_size = max(
        len(indices)
        for indices in client_data
    )

    data_size_norm = (
        data_size
        / max_data_size
    )


    diversity = calculate_data_diversity(
        client_id,
        client_data,
        train_labels
    )


    h = history[client_id]


    r = resources[client_id]


    cpu = normalize_value(
        r["cpu"],
        CPU_MIN,
        CPU_MAX
    )

    battery = normalize_value(
        r["battery"],
        5,
        100
    )

    bandwidth = normalize_value(
        r["bandwidth"],
        BANDWIDTH_MIN,
        BANDWIDTH_MAX
    )

    latency = normalize_value(
        r["latency"],
        LATENCY_MIN,
        LATENCY_MAX
    )




    participation = (
        h["participation_count"]
        / max(NUM_ROUNDS, 1)
    )

    last_epochs = (
        h["last_epochs"]
        / LOCAL_EPOCH_MAX
    )

    last_time = normalize_value(
        h["last_training_time"],
        0,
        30
    )

    last_energy = normalize_value(
        h["last_energy"],
        0,
        30
    )

    last_communication = normalize_value(
        h["last_communication"],
        0,
        10
    )


    state = np.array([


        data_size_norm,
        diversity,
        h["last_loss"],
        h["last_accuracy"],


        cpu,
        battery,
        bandwidth,
        latency,
        r["availability"],

        participation,
        last_epochs,
        last_time,
        last_energy,
        last_communication

    ], dtype=np.float32)


    return state

In [ ]:

class ReplayBuffer:

    def __init__(self, capacity):

        self.buffer = deque(
            maxlen=capacity
        )


    def push(
        self,
        state,
        action,
        reward,
        next_state,
        done
    ):

        self.buffer.append(
            (
                np.asarray(
                    state,
                    dtype=np.float32
                ),

                int(action),

                float(reward),

                np.asarray(
                    next_state,
                    dtype=np.float32
                ),

                float(done)
            )
        )


    def sample(
        self,
        batch_size
    ):

        batch = random.sample(
            self.buffer,
            batch_size
        )

        states, actions, rewards, next_states, dones = zip(
            *batch
        )

        return (
            np.asarray(states),
            np.asarray(actions),
            np.asarray(rewards),
            np.asarray(next_states),
            np.asarray(dones)
        )


    def __len__(self):

        return len(self.buffer)

In [ ]:

class DQNNetwork(nn.Module):

    def __init__(
        self,
        state_dim,
        action_dim
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                state_dim,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                action_dim
            )
        )


    def forward(self, x):

        return self.network(x)

In [ ]:

class DQNAgent:

    def __init__(
        self,
        state_dim,
        action_dim,
        double_dqn=False
    ):

        self.state_dim = state_dim

        self.action_dim = action_dim

        self.double_dqn = double_dqn


        self.gamma = RL_GAMMA

        self.epsilon = EPSILON_START

        self.epsilon_end = EPSILON_END

        self.epsilon_decay = EPSILON_DECAY


        self.policy_net = DQNNetwork(
            state_dim,
            action_dim
        ).to(DEVICE)


        self.target_net = DQNNetwork(
            state_dim,
            action_dim
        ).to(DEVICE)


        self.target_net.load_state_dict(
            self.policy_net.state_dict()
        )

        self.target_net.eval()


        self.optimizer = optim.Adam(
            self.policy_net.parameters(),
            lr=RL_LR
        )


        self.replay_buffer = ReplayBuffer(
            REPLAY_CAPACITY
        )


    def select_action(
        self,
        state,
        training=True
    ):

        if (
            training
            and random.random()
            < self.epsilon
        ):

            return random.randrange(
                self.action_dim
            )


        state_tensor = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)


        with torch.no_grad():

            q_values = self.policy_net(
                state_tensor
            )


        return int(
            q_values.argmax(
                dim=1
            ).item()
        )


    def train_step(self):

        if len(
            self.replay_buffer
        ) < RL_BATCH_SIZE:

            return None


        (
            states,
            actions,
            rewards,
            next_states,
            dones
        ) = self.replay_buffer.sample(
            RL_BATCH_SIZE
        )


        states = torch.tensor(
            states,
            dtype=torch.float32,
            device=DEVICE
        )

        actions = torch.tensor(
            actions,
            dtype=torch.long,
            device=DEVICE
        )

        rewards = torch.tensor(
            rewards,
            dtype=torch.float32,
            device=DEVICE
        )

        next_states = torch.tensor(
            next_states,
            dtype=torch.float32,
            device=DEVICE
        )

        dones = torch.tensor(
            dones,
            dtype=torch.float32,
            device=DEVICE
        )


        current_q = (
            self.policy_net(states)
            .gather(
                1,
                actions.unsqueeze(1)
            )
            .squeeze(1)
        )


        with torch.no_grad():

            if self.double_dqn:

                next_actions = (
                    self.policy_net(
                        next_states
                    )
                    .argmax(
                        dim=1,
                        keepdim=True
                    )
                )

                next_q = (
                    self.target_net(
                        next_states
                    )
                    .gather(
                        1,
                        next_actions
                    )
                    .squeeze(1)
                )

            else:

                next_q = (
                    self.target_net(
                        next_states
                    )
                    .max(
                        dim=1
                    )
                    .values
                )


            target_q = (
                rewards
                + self.gamma
                * next_q
                * (1 - dones)
            )


        loss = F.smooth_l1_loss(
            current_q,
            target_q
        )


        self.optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.policy_net.parameters(),
            5.0
        )

        self.optimizer.step()


        return float(
            loss.item()
        )


    def update_target(self):

        self.target_net.load_state_dict(
            self.policy_net.state_dict()
        )


    def decay_epsilon(self):

        self.epsilon = max(
            self.epsilon_end,
            self.epsilon
            * self.epsilon_decay
        )

In [ ]:


def estimate_communication_cost(
    client_id,
    resources
):

    r = resources[client_id]

    bandwidth = r["bandwidth"]

    latency = r["latency"]



    transmission_time = (
        MODEL_SIZE_MB * 8
        / max(bandwidth, 1e-6)
    )


    latency_time = (
        latency / 1000.0
    )


    total_time = (
        transmission_time
        + latency_time
    )


    return total_time


def estimate_energy_cost(
    client_id,
    epochs,
    training_time,
    resources
):

    cpu = resources[
        client_id
    ]["cpu"]


    energy = (
        training_time
        / max(cpu, 0.1)
    )


    return energy

In [ ]:

def normalize_time(time_value):

    return np.clip(
        time_value / 30.0,
        0.0,
        1.0
    )


def normalize_communication(
    communication
):

    return np.clip(
        communication / 10.0,
        0.0,
        1.0
    )


def normalize_energy(energy):

    return np.clip(
        energy / 30.0,
        0.0,
        1.0
    )

In [ ]:


def calculate_fairness(
    participation_counts
):

    counts = np.asarray(
        participation_counts,
        dtype=np.float64
    )


    if np.sum(counts) == 0:

        return 1.0


    numerator = (
        np.sum(counts) ** 2
    )


    denominator = (
        len(counts)
        * np.sum(counts ** 2)
        + 1e-8
    )



    fairness = (
        numerator
        / denominator
    )


    return float(
        np.clip(
            fairness,
            0,
            1
        )
    )

In [ ]:


def calculate_high_level_reward(

    accuracy_improvement,

    data_utility,

    total_training_time,

    total_communication,

    total_energy,

    fairness

):



    accuracy_component = np.clip(
        accuracy_improvement / 0.10,
        -1.0,
        1.0
    )


    data_component = np.clip(
        data_utility,
        0,
        1
    )


    time_component = normalize_time(
        total_training_time
    )


    communication_component = (
        normalize_communication(
            total_communication
        )
    )


    energy_component = normalize_energy(
        total_energy
    )


    reward = (

        W_GLOBAL_ACCURACY
        * accuracy_component

        + W_DATA_UTILITY
        * data_component

        - W_TIME
        * time_component

        - W_COMMUNICATION
        * communication_component

        - W_ENERGY
        * energy_component

        + W_FAIRNESS
        * fairness

    )


    return float(reward)

In [ ]:

def calculate_low_level_reward(

    local_accuracy_improvement,

    training_time,

    energy,

    communication

):

    accuracy_component = np.clip(
        local_accuracy_improvement
        / 0.10,
        -1.0,
        1.0
    )


    time_component = normalize_time(
        training_time
    )


    energy_component = normalize_energy(
        energy
    )


    communication_component = (
        normalize_communication(
            communication
        )
    )


    reward = (

        W_LOCAL_ACCURACY
        * accuracy_component

        - W_LOCAL_TIME
        * time_component

        - W_LOCAL_ENERGY
        * energy_component

        - W_LOCAL_COMMUNICATION
        * communication_component

    )


    return float(reward)

In [ ]:

def initialize_experiment(
    seed=42
):

    set_seed(seed)


    resources = (
        initialize_client_resources(
            NUM_CLIENTS,
            seed
        )
    )


    history = (
        initialize_client_history()
    )


    global_model = (
        CIFAR10CNN()
        .to(DEVICE)
    )


    return (
        global_model,
        resources,
        history
    )

In [ ]:

STATE_DIM = 14
CLIENT_ACTION_DIM = 2
EPOCH_ACTION_DIM = 5
COUNT_ACTIONS = [5, 8, 10, 12, 15, 20]
COUNT_STATE_DIM = 12
COUNT_ACTION_DIM = len(COUNT_ACTIONS)
RL_BATCH_SIZE = 8
EPISODE_LENGTH = 5
SUCCESS_ACCURACY_THRESHOLD = 50.0
SUCCESS_MIN_MEAN_REWARD = 0.0
CONVERGENCE_ACCURACY_THRESHOLD = 50.0
CONVERGENCE_CONSECUTIVE_ROUNDS = 3
PPO_LR = 3e-4
PPO_CLIP = 0.2
PPO_UPDATE_EPOCHS = 4
PPO_ENTROPY_COEF = 0.01
PPO_VALUE_COEF = 0.5
PPO_MINIBATCH_SIZE = 32
QL_ALPHA = 0.15
QL_EPSILON_START = 1.0
QL_EPSILON_END = 0.05
QL_EPSILON_DECAY = 0.97
QL_GAMMA = RL_GAMMA
print('Seven-method 50-round comparison settings loaded.')


EXTRA_CLIENT_DROPOUT_PROB = 0.0
AVAILABILITY_MULTIPLIER = 1.0


In [ ]:


class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh()
        )
        self.actor = nn.Linear(128, action_dim)
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        z = self.shared(x)
        return self.actor(z), self.critic(z).squeeze(-1)


class PPOAgent:
    def __init__(self, state_dim, action_dim):
        self.model = ActorCritic(state_dim, action_dim).to(DEVICE)
        self.optimizer = optim.Adam(self.model.parameters(), lr=PPO_LR)
        self.memory = []

    def select_action(self, state, training=True, action_mask=None):
        x = torch.as_tensor(
            state, dtype=torch.float32, device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():
            logits, value = self.model(x)

            if action_mask is not None:
                mask = torch.as_tensor(
                    action_mask, dtype=torch.bool, device=DEVICE
                ).unsqueeze(0)
                # The first action is the fallback if fewer than 5 clients
                # are online and no predefined count is feasible.
                if not mask.any():
                    mask[0, 0] = True
                logits = logits.masked_fill(~mask, -1e9)

            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample() if training else torch.argmax(logits, dim=-1)
            log_prob = dist.log_prob(action)

        return int(action.item()), float(log_prob.item()), float(value.item())

    def store(self, state, action, log_prob, reward, next_state, done, action_mask=None):
        mask = None if action_mask is None else np.asarray(action_mask, dtype=bool)
        self.memory.append((np.asarray(state, dtype=np.float32), int(action),
                            float(log_prob), float(reward),
                            np.asarray(next_state, dtype=np.float32), float(done), mask))

    def update(self):
        if len(self.memory) < 2:
            return None

        states, actions, old_log_probs, rewards, next_states, dones, masks = zip(*self.memory)

        states = torch.as_tensor(np.asarray(states), dtype=torch.float32, device=DEVICE)
        actions = torch.as_tensor(actions, dtype=torch.long, device=DEVICE)
        old_log_probs = torch.as_tensor(old_log_probs, dtype=torch.float32, device=DEVICE)
        rewards = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
        next_states = torch.as_tensor(np.asarray(next_states), dtype=torch.float32, device=DEVICE)
        dones = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE)

        with torch.no_grad():
            _, values = self.model(states)
            _, next_values = self.model(next_states)
            td_targets = rewards + RL_GAMMA * next_values * (1.0 - dones)
            advantages = td_targets - values
            advantages = (advantages - advantages.mean()) / (
                advantages.std(unbiased=False) + 1e-8
            )

        n = len(self.memory)
        losses = []

        for _ in range(PPO_UPDATE_EPOCHS):
            order = torch.randperm(n, device=DEVICE)
            for start in range(0, n, PPO_MINIBATCH_SIZE):
                ids = order[start:start + PPO_MINIBATCH_SIZE]
                logits, predicted_values = self.model(states[ids])
                if any(m is not None for m in masks):
                    mask_batch = torch.as_tensor(np.asarray(masks), dtype=torch.bool, device=DEVICE)[ids]
                    logits = logits.masked_fill(~mask_batch, -1e9)
                dist = torch.distributions.Categorical(logits=logits)
                new_log_probs = dist.log_prob(actions[ids])
                entropy = dist.entropy().mean()

                ratios = torch.exp(new_log_probs - old_log_probs[ids])
                unclipped = ratios * advantages[ids]
                clipped = torch.clamp(
                    ratios, 1.0 - PPO_CLIP, 1.0 + PPO_CLIP
                ) * advantages[ids]

                policy_loss = -torch.min(unclipped, clipped).mean()
                value_loss = F.mse_loss(predicted_values, td_targets[ids])
                loss = (
                    policy_loss
                    + PPO_VALUE_COEF * value_loss
                    - PPO_ENTROPY_COEF * entropy
                )

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
                self.optimizer.step()
                losses.append(float(loss.item()))

        self.memory.clear()
        return float(np.mean(losses)) if losses else None


def make_agent(algorithm, state_dim, action_dim):
    algorithm = algorithm.upper()
    if algorithm == "PPO":
        return PPOAgent(state_dim, action_dim)
    if algorithm in ("DQN", "DDQN"):
        return DQNAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            double_dqn=(algorithm == "DDQN")
        )
    raise ValueError(f"Unsupported algorithm: {algorithm}")


In [ ]:

def build_count_state(resources, history, global_accuracy, round_idx, total_rounds):
    vals = list(resources.values())
    cpu = np.asarray([v["cpu"] for v in vals], dtype=np.float32)
    battery = np.asarray([v["battery"] for v in vals], dtype=np.float32)
    bandwidth = np.asarray([v["bandwidth"] for v in vals], dtype=np.float32)
    latency = np.asarray([v["latency"] for v in vals], dtype=np.float32)
    availability = np.asarray([v["availability"] for v in vals], dtype=np.float32)
    participation = np.asarray(
        [history[i]["participation_count"] for i in range(NUM_CLIENTS)],
        dtype=np.float32
    )
    last_acc = np.asarray(
        [history[i]["last_accuracy"] for i in range(NUM_CLIENTS)],
        dtype=np.float32
    )

    state = np.asarray([
        round_idx / max(total_rounds, 1),
        global_accuracy / 100.0,
        cpu.mean() / CPU_MAX,
        cpu.std() / CPU_MAX,
        battery.mean() / 100.0,
        bandwidth.mean() / BANDWIDTH_MAX,
        bandwidth.std() / BANDWIDTH_MAX,
        latency.mean() / LATENCY_MAX,
        availability.mean(),
        participation.mean() / max(total_rounds, 1),
        calculate_fairness(participation),
        last_acc.mean()
    ], dtype=np.float32)

    return np.nan_to_num(state, nan=0.0, posinf=1.0, neginf=0.0)


def choose_count(agent, algorithm, state, available_count, training=True):
    mask = np.asarray(
        [n <= available_count for n in COUNT_ACTIONS],
        dtype=bool
    )
    if not mask.any():
        mask[0] = True

    if algorithm == "PPO":
        action, log_prob, value = agent.select_action(
            state, training=training, action_mask=mask
        )
        requested = COUNT_ACTIONS[action]
        actual = min(requested, available_count)
        return action, requested, actual, log_prob, value

    valid = np.flatnonzero(mask)
    if algorithm == "QLEARNING":
        action = agent.select_action(state, training=training)
        if action not in valid:
            action = int(valid[np.argmax(agent.q_values(state)[valid])])
        requested = COUNT_ACTIONS[action]
        return action, requested, min(requested, available_count), None, None
    if training and random.random() < agent.epsilon:
        action = int(random.choice(valid.tolist()))
    else:
        x = torch.as_tensor(
            state, dtype=torch.float32, device=DEVICE
        ).unsqueeze(0)
        with torch.no_grad():
            q = agent.policy_net(x)[0].detach().cpu().numpy()
        action = int(max(valid, key=lambda i: q[i]))

    requested = COUNT_ACTIONS[action]
    return action, requested, min(requested, available_count), None, None


def build_selection_matrix(selected_history):
    matrix = np.zeros((NUM_CLIENTS, len(selected_history)))
    for r, clients in enumerate(selected_history):
        for cid in clients:
            matrix[cid, r] = 1
    return matrix


def build_epoch_matrix(epoch_history):
    matrix = np.zeros((NUM_CLIENTS, len(epoch_history)))
    for r, assignments in enumerate(epoch_history):
        for cid, epochs in assignments.items():
            matrix[cid, r] = epochs
    return matrix


In [ ]:
# ============================================================
# TABULAR Q-LEARNING AGENT (discretized continuous client/global state)
# ============================================================
class QLearningAgent:
    def __init__(self, state_dim, action_dim):
        self.state_dim, self.action_dim = state_dim, action_dim
        self.q = defaultdict(lambda: np.zeros(action_dim, dtype=np.float32))
        self.alpha, self.gamma = QL_ALPHA, QL_GAMMA
        self.epsilon = QL_EPSILON_START
        self.memory = deque(maxlen=REPLAY_CAPACITY)
    def key(self, state):
        x = np.asarray(state, dtype=np.float32)
        # 4-bin discretization per dimension; sparse dictionary avoids allocating the full grid.
        return tuple(np.clip(np.floor(np.nan_to_num(x) * 4), 0, 3).astype(int).tolist())
    def q_values(self, state): return self.q[self.key(state)]
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon: return random.randrange(self.action_dim)
        return int(np.argmax(self.q_values(state)))
    def push(self, state, action, reward, next_state, done):
        self.memory.append((self.key(state), int(action), float(reward), self.key(next_state), bool(done)))
    def train_step(self):
        if not self.memory: return None
        batch = random.sample(self.memory, min(RL_BATCH_SIZE, len(self.memory)))
        losses=[]
        for state, action, reward, nxt, done in batch:
            target = reward if done else reward + self.gamma * float(np.max(self.q[nxt]))
            old = self.q[state][action]
            self.q[state][action] += self.alpha * (target-old)
            losses.append((target-old)**2)
        return float(np.mean(losses))
    def decay_epsilon(self): self.epsilon=max(QL_EPSILON_END, self.epsilon*QL_EPSILON_DECAY)
    def update_target(self): pass

def make_agent(algorithm, state_dim, action_dim):
    algorithm = algorithm.upper()
    if algorithm == 'PPO': return PPOAgent(state_dim, action_dim)
    if algorithm == 'QLEARNING': return QLearningAgent(state_dim, action_dim)
    if algorithm in ('DQN','DDQN'):
        return DQNAgent(state_dim, action_dim, double_dqn=(algorithm == 'DDQN'))
    raise ValueError(f'Unsupported algorithm: {algorithm}')



In [ ]:


def make_malicious_client_set(seed):
    rng = np.random.default_rng(seed + 771)
    n_bad = max(1, int(round(NUM_CLIENTS * MALICIOUS_CLIENT_FRACTION))) if ENABLE_MALICIOUS_CLIENT_SIMULATION else 0
    return set(rng.choice(NUM_CLIENTS, size=n_bad, replace=False).tolist())


def inject_sign_flip_update(local_state, global_state, strength=1.0):
    poisoned = {}
    for key, value in local_state.items():
        base = global_state[key]
        if torch.is_floating_point(value):

            poisoned[key] = base + (-float(strength) * (value - base))
        else:
            poisoned[key] = value.clone()
    return poisoned


def _state_delta_vector(local_state, global_state):
    pieces = []
    for key, value in local_state.items():
        if torch.is_floating_point(value):
            pieces.append((value.detach().float().cpu() - global_state[key].detach().float().cpu()).reshape(-1))
    return torch.cat(pieces) if pieces else torch.zeros(1)


def detect_suspicious_updates(client_states, global_state,
                              cosine_threshold=DETECTOR_COSINE_THRESHOLD,
                              max_norm_ratio=DETECTOR_MAX_NORM_RATIO):
    """Median-reference detector using update cosine and relative update norm."""
    if len(client_states) < 3:
        return set(), {cid: {"cosine": np.nan, "norm_ratio": np.nan, "flagged": False}
                       for cid, _ in client_states}
    vectors = {cid: _state_delta_vector(state, global_state) for cid, state in client_states}
    matrix = torch.stack(list(vectors.values()))
    reference = torch.median(matrix, dim=0).values
    ref_norm = torch.linalg.vector_norm(reference).item() + 1e-12
    ref_unit = reference / ref_norm
    flagged, diagnostics = set(), {}
    for cid, vec in vectors.items():
        norm = torch.linalg.vector_norm(vec).item()
        cosine = torch.nn.functional.cosine_similarity(vec.unsqueeze(0), ref_unit.unsqueeze(0)).item()
        ratio = norm / ref_norm
        is_bad = (cosine < cosine_threshold) or (ratio > max_norm_ratio)
        diagnostics[cid] = {"cosine": float(cosine), "norm_ratio": float(ratio), "flagged": bool(is_bad)}
        if is_bad:
            flagged.add(cid)
    return flagged, diagnostics




In [ ]:

METHODS = {
    "QLEARNING3": ("QLEARNING", "QLEARNING", "QLEARNING"),
    "DQN3": ("DQN", "DQN", "DQN"),
    "DDQN3": ("DDQN", "DDQN", "DDQN"),
    "PPO3": ("PPO", "PPO", "PPO"),
    "DQN_PPO": ("DQN", "DQN", "PPO"),
    "PPO_DQN": ("PPO", "DQN", "DQN")
}

def _agent_score(agent, algo, state):
    if algo == "PPO":
        x=torch.as_tensor(state,dtype=torch.float32,device=DEVICE).unsqueeze(0)
        with torch.no_grad(): logits,_=agent.model(x)
        probs=torch.softmax(logits,dim=-1)[0]
        return float((probs[1]-probs[0]).item())
    if algo == "QLEARNING": return float(agent.q_values(state)[1]-agent.q_values(state)[0])
    x=torch.as_tensor(state,dtype=torch.float32,device=DEVICE).unsqueeze(0)
    with torch.no_grad(): q=agent.policy_net(x)[0]
    return float((q[1]-q[0]).item())

def run_three_decision_experiment(method="DQN3", num_rounds=50, seed=42, verbose=True, ablation_mode=None):
    method=method.upper()
    if method not in list(METHODS)+["FEDAVG"]: raise ValueError(method)
    set_seed(seed)
    global_model, resources, history=initialize_experiment(seed)
    global_state=get_model_state(global_model)
    if method == "FEDAVG":
        count_algo=selection_algo=epoch_algo="FIXED"
        count_agent=selection_agent=epoch_agent=None
    else:
        count_algo,selection_algo,epoch_algo=METHODS[method]

        set_seed(seed + {"QLEARNING3":101,"DQN3":201,"DDQN3":301,"PPO3":401,"DQN_PPO":501,"PPO_DQN":601}[method])
        count_agent=make_agent(count_algo,COUNT_STATE_DIM,COUNT_ACTION_DIM)
        selection_agent=make_agent(selection_algo,STATE_DIM,NUM_CLIENTS if selection_algo=="PPO" else CLIENT_ACTION_DIM)
        epoch_agent=make_agent(epoch_algo,STATE_DIM,EPOCH_ACTION_DIM)

        set_seed(seed+9000)
    results=[]; selected_history=[]; epoch_history=[]; count_history=[]
    malicious_clients = make_malicious_client_set(seed)
    total_detected=total_true_positive=total_false_positive=total_malicious_selected=0
    total_selected_before_dropout=total_dropped_after_selection=0
    previous_global_accuracy=10.0; episode_return=0.0; episode_rewards=[]
    convergence_hits=0; convergence_round=None; cumulative_interactions=0
    for round_idx in range(num_rounds):
        round_start=time.time(); rl_start=time.time()
        client_states={cid:build_client_state(cid,client_data,train_labels,resources,history) for cid in range(NUM_CLIENTS)}
        available=[cid for cid in range(NUM_CLIENTS) if random.random()<min(1.0, resources[cid]["availability"]*AVAILABILITY_MULTIPLIER) and resources[cid]["battery"]>5.0 and random.random()>=EXTRA_CLIENT_DROPOUT_PROB]
        if not available: available=[max(resources,key=lambda cid:resources[cid]["availability"])]
        count_state=build_count_state(resources,history,previous_global_accuracy,round_idx,num_rounds)
        count_log_prob=None; count_action=None
        if method == "FEDAVG": requested_count=10; actual_count=min(requested_count,len(available))
        else:
            if count_algo=="PPO":
                mask=np.asarray([n<=len(available) for n in COUNT_ACTIONS],dtype=bool)
                count_action,count_log_prob,_=count_agent.select_action(count_state,True,mask)
                requested_count=COUNT_ACTIONS[count_action]; actual_count=min(requested_count,len(available))
            else:
                count_action=choose_count(count_agent,count_algo,count_state,len(available),True)[0]
                requested_count=COUNT_ACTIONS[count_action]; actual_count=min(requested_count,len(available))
        if ablation_mode in ("FIXED_COUNT", "ALL_FIXED"):
            requested_count=10; actual_count=min(requested_count,len(available))

        if method=="FEDAVG" or ablation_mode in ("RANDOM_SELECTION", "ALL_FIXED"):
            selected=random.sample(available,actual_count)
        else:
            if selection_algo == "PPO":

                selected=[]; selection_rollout=[]; remaining=set(available)
                selection_state=np.mean(np.stack([client_states[c] for c in available]),axis=0).astype(np.float32)
                for _ in range(actual_count):
                    mask=np.zeros(NUM_CLIENTS,dtype=bool)
                    mask[list(remaining)]=True
                    action, lp, _ = selection_agent.select_action(
                        selection_state, True, action_mask=mask
                    )
                    cid=int(action); selected.append(cid)
                    selection_rollout.append((cid, lp, mask.copy()))
                    remaining.remove(cid)
            else:
                scored=[]
                for cid in available:
                    score=_agent_score(selection_agent,selection_algo,client_states[cid])
                    if selection_algo in ("DQN","DDQN","QLEARNING") and random.random()<selection_agent.epsilon:
                        score += float(np.random.normal(0,0.5))
                    scored.append((cid,score))
                scored.sort(key=lambda z:z[1],reverse=True)
                selected=[cid for cid,_ in scored[:actual_count]]

        selected_before_dropout = selected.copy()
        total_selected_before_dropout += len(selected_before_dropout)
        if ENABLE_POST_SELECTION_DROPOUT and POST_SELECTION_DROPOUT_PROB > 0:
            selected = [cid for cid in selected_before_dropout
                        if random.random() >= POST_SELECTION_DROPOUT_PROB]
        total_dropped_after_selection += len(selected_before_dropout) - len(selected)
        epoch_assignments={}; epoch_info={}
        for cid in selected:
            if method=="FEDAVG" or ablation_mode in ("FIXED_EPOCHS", "ALL_FIXED"): action=1; epoch_assignments[cid]=2; epoch_info[cid]=None
            elif epoch_algo=="PPO":
                action,lp,val=epoch_agent.select_action(client_states[cid],True); epoch_info[cid]=(action,lp,val); epoch_assignments[cid]=LOCAL_EPOCH_MIN+action
            else:
                action=epoch_agent.select_action(client_states[cid],True); epoch_info[cid]=None; epoch_assignments[cid]=LOCAL_EPOCH_MIN+action
        rl_decision_time=time.time()-rl_start
        local_states=[]; local_sizes=[]; utilities=[]; local_rewards={}
        compute=comm=energy=0.0
        for cid in selected:
            epochs=epoch_assignments[cid]; old_acc=history[cid]["last_accuracy"]
            lr=train_local_model(global_state,cid,epochs,client_data,fl_train_dataset)
            update_state = lr["state"]
            is_malicious = ENABLE_MALICIOUS_CLIENT_SIMULATION and cid in malicious_clients
            if is_malicious:
                update_state = inject_sign_flip_update(update_state, global_state, MALICIOUS_SIGN_FLIP_STRENGTH)
                total_malicious_selected += 1
            local_states.append(update_state); local_sizes.append(lr["samples"])
            cp=estimate_communication_cost(cid,resources); ep=estimate_energy_cost(cid,epochs,lr["training_time"],resources)
            compute+=lr["training_time"]; comm+=cp; energy+=ep
            utility=calculate_data_diversity(cid,client_data,train_labels); utilities.append(utility)
            local_reward=calculate_low_level_reward(lr["accuracy"]-old_acc,lr["training_time"],ep,cp); local_rewards[cid]=float(local_reward)
            history[cid].update({"previous_accuracy":old_acc,"last_accuracy":lr["accuracy"],"last_loss":lr["loss"],"last_epochs":epochs,"last_training_time":lr["training_time"],"last_energy":ep,"last_communication":cp,"participation_count":history[cid]["participation_count"]+1})
            nxt=build_client_state(cid,client_data,train_labels,resources,history); a=epochs-LOCAL_EPOCH_MIN
            if method!="FEDAVG":
                if epoch_algo=="PPO" and ablation_mode != "FIXED_EPOCHS":
                    _,lp,_=epoch_info[cid]; epoch_agent.store(client_states[cid],a,lp,local_reward,nxt,round_idx==num_rounds-1)
                elif epoch_algo=="QLEARNING": epoch_agent.push(client_states[cid],a,local_reward,nxt,round_idx==num_rounds-1)
                else: epoch_agent.replay_buffer.push(client_states[cid],a,local_reward,nxt,round_idx==num_rounds-1)

        detected_clients=set(); detector_diagnostics={}
        if local_states and ENABLE_MALICIOUS_CLIENT_SIMULATION:
            candidate_updates = list(zip(selected, local_states))
            detected_clients, detector_diagnostics = detect_suspicious_updates(candidate_updates, global_state)
            total_detected += len(detected_clients)
            total_true_positive += len(detected_clients.intersection(malicious_clients))
            total_false_positive += len(detected_clients.difference(malicious_clients))
            keep_idx = [i for i, cid in enumerate(selected) if cid not in detected_clients]

            if keep_idx:
                agg_states=[local_states[i] for i in keep_idx]
                agg_sizes=[local_sizes[i] for i in keep_idx]
            else:
                agg_states, agg_sizes = local_states, local_sizes
        else:
            agg_states, agg_sizes = local_states, local_sizes
        if agg_states:
            global_state=fedavg(agg_states,agg_sizes); load_model_state(global_model,global_state)
        validation_loss,validation_acc=evaluate_model(global_model, validation_loader)
        fairness=calculate_fairness([history[i]["participation_count"] for i in range(NUM_CLIENTS)])
        mean_utility=float(np.mean(utilities)) if utilities else 0.0
        high_reward=calculate_high_level_reward((validation_acc-previous_global_accuracy)/100.0,mean_utility,compute,comm,energy,fairness)
        selected_set=set(selected); contribution={cid:0.5*np.clip((history[cid]["last_accuracy"]-history[cid]["previous_accuracy"])/0.10,-1,1)+0.2*np.clip(calculate_data_diversity(cid,client_data,train_labels),0,1) for cid in selected}
        mean_contrib=float(np.mean(list(contribution.values()))) if contribution else 0.0
        if method!="FEDAVG":
            for cid in range(NUM_CLIENTS):
                nxt=build_client_state(cid,client_data,train_labels,resources,history); a=int(cid in selected_set)
                rew=(high_reward+0.25*(contribution[cid]-mean_contrib)) if a else 0.0
                if selection_algo=="PPO":

                    pass
                elif selection_algo=="QLEARNING": selection_agent.push(client_states[cid],a,float(rew),nxt,round_idx==num_rounds-1)
                else: selection_agent.replay_buffer.push(client_states[cid],a,float(rew),nxt,round_idx==num_rounds-1)
            if selection_algo=="PPO" and ablation_mode != "RANDOM_SELECTION":
                for cid, lp, selection_mask in selection_rollout:
                    nxt=build_client_state(cid,client_data,train_labels,resources,history)
                    rew=float(high_reward + 0.25*(contribution.get(cid,0.0)-mean_contrib))
                    selection_agent.store(selection_state,cid,lp,rew,selection_state,
                                          round_idx==num_rounds-1,action_mask=selection_mask)
            nxt_count=build_count_state(resources,history,validation_acc,round_idx+1,num_rounds)
            if count_algo=="PPO": count_agent.store(count_state,count_action,count_log_prob,high_reward,nxt_count,round_idx==num_rounds-1)
            elif count_algo=="QLEARNING": count_agent.push(count_state,count_action,high_reward,nxt_count,round_idx==num_rounds-1)
            else: count_agent.replay_buffer.push(count_state,count_action,high_reward,nxt_count,round_idx==num_rounds-1)
        update_start=time.time(); losses={"count":None,"selection":None,"epoch":None}
        if method!="FEDAVG":
            for name,agent,algo in [("count",count_agent,count_algo),("selection",selection_agent,selection_algo),("epoch",epoch_agent,epoch_algo)]:
                disabled = (ablation_mode == "ALL_FIXED" or
                            (name=="count" and ablation_mode=="FIXED_COUNT") or
                            (name=="selection" and ablation_mode=="RANDOM_SELECTION") or
                            (name=="epoch" and ablation_mode=="FIXED_EPOCHS"))
                if disabled:
                    continue
                if algo=="PPO":
                    if (round_idx+1)%EPISODE_LENGTH==0 or round_idx==num_rounds-1: losses[name]=agent.update()
                else:
                    losses[name]=agent.train_step()
                    if algo in ("DQN","DDQN") and (round_idx+1)%TARGET_UPDATE_FREQ==0: agent.update_target()
                    agent.decay_epsilon()
        rl_update_time=time.time()-update_start
        update_client_resources(resources,selected)
        episode_return+=float(high_reward); episode_rewards.append(float(high_reward))
        episode_end=(round_idx+1)%EPISODE_LENGTH==0 or round_idx==num_rounds-1
        ep_return=episode_return if episode_end else np.nan
        ep_success=int(validation_acc>=SUCCESS_ACCURACY_THRESHOLD and np.mean(episode_rewards)>SUCCESS_MIN_MEAN_REWARD) if episode_end else np.nan
        if episode_end: episode_return=0.0; episode_rewards=[]
        convergence_hits=convergence_hits+1 if validation_acc>=CONVERGENCE_ACCURACY_THRESHOLD else 0
        if convergence_round is None and convergence_hits>=CONVERGENCE_CONSECUTIVE_ROUNDS: convergence_round=round_idx+1
        cumulative_interactions+=(1 + len(available) + len(selected)) if method!="FEDAVG" else 0
        row={"round":round_idx+1,"validation_accuracy":float(validation_acc),"validation_loss":float(validation_loss),"accuracy":float(validation_acc),"loss":float(validation_loss),"reward":float(high_reward),"cumulative_reward":float(sum(x["reward"] for x in results)+high_reward),"episode_return_at_end":ep_return,"episode_success_at_end":ep_success,"requested_count":int(requested_count),"selected_count":len(selected),"available_count":len(available),"selected_before_dropout":len(selected_before_dropout),
              "post_selection_dropped_count":len(selected_before_dropout)-len(selected),
              "malicious_selected_count":sum(cid in malicious_clients for cid in selected),
              "malicious_detected_count":len(detected_clients),
              "detector_true_positive_count":len(detected_clients.intersection(malicious_clients)),
              "detector_false_positive_count":len(detected_clients.difference(malicious_clients)),
              "mean_detector_cosine":float(np.nanmean([v["cosine"] for v in detector_diagnostics.values()])) if detector_diagnostics else np.nan,
              "mean_epochs":float(np.mean(list(epoch_assignments.values()))) if epoch_assignments else 0.0,"fairness":float(fairness),"data_utility":mean_utility,"sum_local_compute_time_sec":compute,"communication_time_proxy_sec":comm,"modeled_communication_mb":len(selected)*2.0*MODEL_SIZE_MB,"sequential_local_compute_sec":compute,"simulated_round_wall_time_sec":max([history[c]["last_training_time"] for c in selected], default=0.0)+comm,"energy_proxy_units":energy,"rl_decision_time_sec":rl_decision_time if method!="FEDAVG" else 0.0,"rl_update_time_sec":rl_update_time,"cumulative_agent_interactions":cumulative_interactions,"selection_decisions_this_round":len(available),"epoch_decisions_this_round":len(selected),"count_loss":losses["count"],"selection_loss":losses["selection"],"epoch_loss":losses["epoch"],"convergence_round":convergence_round,"wall_time_sec":time.time()-round_start}
        results.append(row); selected_history.append(selected.copy()); epoch_history.append(epoch_assignments.copy()); count_history.append(int(requested_count)); previous_global_accuracy=validation_acc
        if verbose: print(f"{method} | Round {round_idx+1:02d}/{num_rounds} | Acc={validation_acc:.2f}% | Count={len(selected)} | MeanEpoch={row['mean_epochs']:.2f} | Reward={high_reward:.4f}")
    df=pd.DataFrame(results); eps=df["episode_return_at_end"].dropna(); succ=df["episode_success_at_end"].dropna()
    summary={"method":(method if ablation_mode is None else f"{method}_{ablation_mode}"),"final_accuracy":float(df.accuracy.iloc[-1]),"best_accuracy":float(df.accuracy.max()),"mean_episodic_return":float(eps.mean()) if len(eps) else np.nan,"success_rate_percent":float(100*succ.mean()) if len(succ) else np.nan,"convergence_round":convergence_round,"total_reward":float(df.reward.sum()),"total_rl_decision_time_sec":float(df.rl_decision_time_sec.sum()),"total_rl_update_time_sec":float(df.rl_update_time_sec.sum()),"total_local_compute_time_sec":float(df.sum_local_compute_time_sec.sum()),"total_communication_time_proxy_sec":float(df.communication_time_proxy_sec.sum()),"total_energy_proxy_units":float(df.energy_proxy_units.sum()),"mean_fairness":float(df.fairness.mean()),"sample_efficiency_final_reward_per_interaction":float(df.reward.sum()/max(cumulative_interactions,1)),"cumulative_accuracy_regret_vs_fedavg":np.nan,
              "selected_before_dropout_total":int(total_selected_before_dropout),
              "post_selection_dropout_total":int(total_dropped_after_selection),
              "malicious_clients_per_run":int(len(malicious_clients)),
              "malicious_selected_total":int(total_malicious_selected),
              "malicious_updates_detected_total":int(total_detected),
              "malicious_detection_true_positives":int(total_true_positive),
              "malicious_detection_false_positives":int(total_false_positive),
              "malicious_detection_precision":float(total_true_positive/max(total_detected,1)),
              "malicious_detection_recall":float(total_true_positive/max(total_malicious_selected,1))}
    return {"method":(method if ablation_mode is None else f"{method}_{ablation_mode}"),"results":df,"summary":summary,"final_model_state":copy.deepcopy(global_state),"selected_history":selected_history,"epoch_history":epoch_history,"count_history":count_history}


## Run instructions (improved version)
1. Run cells 0–37 in order, including Drive mount and CIFAR-10 loading.
2. Run the 2-round smoke test first.
3. For a quick comparison use one seed; for final reporting use at least 3 seeds (preferably 5).
4. Do not change the validation/test split after inspecting final test results.
5. The official test set is evaluated only in the final evaluation cell after all training runs.
6. Outputs are saved under `improved_seven_method_results/`.


In [ ]:

RUN_METHODS = [
    'FEDAVG',
    'QLEARNING3',
    'DQN3',
    'DDQN3',
    'PPO3',
    'DQN_PPO',
    'PPO_DQN'
]

RUN_SEEDS = [42]

EXPERIMENT_ROUNDS = 50

experiment_results = {}
all_run_summaries = []

total_experiments = len(RUN_METHODS) * len(RUN_SEEDS)
completed_experiments = 0

import time

overall_start = time.time()

for seed in RUN_SEEDS:

    for method_name in RUN_METHODS:

        completed_experiments += 1

        key = f'{method_name}_seed{seed}'

        print("\n" + "="*75, flush=True)
        print(
            f"STARTING EXPERIMENT {completed_experiments}/{total_experiments}",
            flush=True
        )
        print(f"METHOD: {method_name}", flush=True)
        print(f"SEED: {seed}", flush=True)
        print(f"TOTAL ROUNDS: {EXPERIMENT_ROUNDS}", flush=True)
        print("="*75, flush=True)

        method_start = time.time()

        experiment_results[key] = run_three_decision_experiment(
            method=method_name,
            num_rounds=EXPERIMENT_ROUNDS,
            seed=seed,
            verbose=True
        )

        method_time = time.time() - method_start

        sm = dict(experiment_results[key]['summary'])
        sm['seed'] = seed
        sm['method_time_sec'] = method_time

        all_run_summaries.append(sm)

        print("\n" + "-"*75, flush=True)
        print(f"COMPLETED: {method_name}", flush=True)
        print(
            f"Final Validation Accuracy: {sm['final_accuracy']:.2f}%",
            flush=True
        )
        print(
            f"Best Validation Accuracy: {sm['best_accuracy']:.2f}%",
            flush=True
        )
        print(
            f"Mean Episodic Return: {sm['mean_episodic_return']:.4f}",
            flush=True
        )
        print(
            f"Success Rate: {sm['success_rate_percent']:.2f}%",
            flush=True
        )
        print(
            f"Experiment Time: {method_time/60:.2f} minutes",
            flush=True
        )
        print("-"*75, flush=True)


        summary_df = pd.DataFrame(all_run_summaries)

        display(
            summary_df[
                [
                    'method',
                    'seed',
                    'final_accuracy',
                    'best_accuracy',
                    'mean_episodic_return',
                    'success_rate_percent',
                    'convergence_round'
                ]
            ]
        )

print("\n" + "="*75)
print("ALL EXPERIMENTS COMPLETED!")
print(f"Total time: {(time.time()-overall_start)/60:.2f} minutes")
print("="*75)

summary_df = pd.DataFrame(all_run_summaries)

display(summary_df)

In [ ]:

OUT_DIR='seven_method_cifar10_results'; os.makedirs(OUT_DIR,exist_ok=True)
summary_df.to_csv(f'{OUT_DIR}/architecture_summary_all_seeds.csv',index=False)
for key,out in experiment_results.items(): out['results'].to_csv(f'{OUT_DIR}/{key}_round_metrics.csv',index=False)

def plot_metric(metric,ylabel,title,filename):
    plt.figure(figsize=(11,5))
    for key,out in experiment_results.items(): plt.plot(out['results']['round'],out['results'][metric],alpha=.8,label=key)
    plt.xlabel('FL Round'); plt.ylabel(ylabel); plt.title(title); plt.grid(alpha=.3); plt.legend(fontsize=7); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{filename}',dpi=160); plt.show()
for m,y,t,f in [('accuracy','Global test accuracy (%)','Global test accuracy','accuracy.png'),('reward','Reward','Per-round RL reward','reward.png'),('cumulative_reward','Cumulative reward','Cumulative reward','cumulative_reward.png'),('fairness','Jain fairness index','Participation fairness','fairness.png'),('sum_local_compute_time_sec','Local compute seconds','Local computation per round','local_compute.png'),('communication_time_proxy_sec','Communication time proxy','Communication proxy per round','communication_proxy.png'),('energy_proxy_units','Energy proxy units','Energy proxy per round','energy_proxy.png'),('rl_decision_time_sec','Seconds','RL decision overhead','rl_decision_time.png'),('rl_update_time_sec','Seconds','RL update overhead','rl_update_time.png'),('cumulative_agent_interactions','Interactions','Sample efficiency: cumulative reward vs interactions','interactions.png')]: plot_metric(m,y,t,f)


In [ ]:

episode_rows=[]
for key,out in experiment_results.items():
    d=out['results'];
    for _,r in d[d.episode_return_at_end.notna()].iterrows(): episode_rows.append({'run':key,'episode':int((r['round']-1)//EPISODE_LENGTH+1),'episodic_return':r['episode_return_at_end'],'success':r['episode_success_at_end']})
episode_metrics_df=pd.DataFrame(episode_rows); episode_metrics_df.to_csv(f'{OUT_DIR}/episode_metrics.csv',index=False); display(episode_metrics_df)

for seed in RUN_SEEDS:
    base_key=f'FEDAVG_seed{seed}'
    if base_key in experiment_results:
        base=experiment_results[base_key]['results'].set_index('round')['accuracy']
        for method in RUN_METHODS:
            key=f'{method}_seed{seed}'; d=experiment_results[key]['results']
            if method=='FEDAVG': reg=np.zeros(len(d))
            else: reg=np.maximum(0.0,base.values-d['accuracy'].values)
            d['accuracy_regret_vs_fedavg']=reg; d['cumulative_accuracy_regret_vs_fedavg']=np.cumsum(reg)
            experiment_results[key]['summary']['cumulative_accuracy_regret_vs_fedavg']=float(np.sum(reg))

def evaluate_train_accuracy(model):
    loader=DataLoader(fl_train_dataset,batch_size=256,shuffle=False,num_workers=2)
    model.eval(); correct=total=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE); y=y.to(DEVICE); pred=model(x).argmax(1); correct+=(pred==y).sum().item(); total+=y.numel()
    return 100.0*correct/max(total,1)

def _train_accuracy_from_state(state):
    m=CIFAR10CNN().to(DEVICE); load_model_state(m,state)
    loader=DataLoader(fl_train_dataset,batch_size=256,shuffle=False,num_workers=2)
    m.eval(); correct=total=0
    with torch.no_grad():
        for xx,yy in loader:
            xx=xx.to(DEVICE); yy=yy.to(DEVICE); pp=m(xx).argmax(1); correct+=(pp==yy).sum().item(); total+=yy.numel()
    return 100.0*correct/max(total,1)
summary_rows=[]
for k,o in experiment_results.items():
    row=dict(o['summary'],run=k)
    tr_acc=_train_accuracy_from_state(o['final_model_state'])
    row['train_accuracy_pct']=tr_acc
    row['train_validation_gap_pct']=tr_acc-row['final_accuracy']
    summary_rows.append(row)
summary_df=pd.DataFrame(summary_rows)
summary_df['oracle_cumulative_regret']=np.nan
summary_df.to_csv(f'{OUT_DIR}/architecture_summary_with_rl_metrics.csv',index=False)

if len(RUN_SEEDS)>=2:
    stability=summary_df.groupby('method').agg(final_accuracy_mean=('final_accuracy','mean'),final_accuracy_std=('final_accuracy','std'),mean_return_mean=('mean_episodic_return','mean'),mean_return_std=('mean_episodic_return','std'),fairness_mean=('mean_fairness','mean'),fairness_std=('mean_fairness','std')).reset_index()
else:
    stability=pd.DataFrame({'note':['Only one seed run: stability across seeds cannot be estimated. Use at least 3 seeds.']})
stability.to_csv(f'{OUT_DIR}/seed_stability.csv',index=False); display(stability)

for col in ['count_loss','selection_loss','epoch_loss']:
    plt.figure(figsize=(10,4))
    for key,o in experiment_results.items(): plt.plot(o['results']['round'],o['results'][col],label=key)
    plt.title(col); plt.xlabel('Round'); plt.ylabel('Loss'); plt.grid(alpha=.3); plt.legend(fontsize=7); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{col}.png',dpi=150); plt.show()
print('RL metrics included: episodic return, mean episodic return, success rate, cumulative reward, benchmark-relative cumulative accuracy regret, convergence round, sample efficiency/interactions, policy decision/update time, policy losses, and seed stability (when multiple seeds are run).')
print('Oracle regret is NA because no exact oracle policy is defined. The reported train-validation gap is not a test-set generalization gap.')


In [ ]:

def evaluate_state_on_loader(state, loader):
    model = CIFAR10CNN().to(DEVICE)
    load_model_state(model, state)
    return evaluate_model(model, loader)

final_test_rows = []
for run_key, run_output in experiment_results.items():
    test_loss, test_accuracy = evaluate_state_on_loader(run_output["final_model_state"], test_loader)
    run_output["summary"]["final_test_accuracy"] = float(test_accuracy)
    run_output["summary"]["final_test_loss"] = float(test_loss)
    final_test_rows.append({"run": run_key, "method": run_output["method"],
                            "final_test_accuracy": test_accuracy, "final_test_loss": test_loss})
final_test_df = pd.DataFrame(final_test_rows)
final_test_df.to_csv(f"{OUT_DIR}/final_test_only_metrics.csv", index=False)
display(final_test_df)


In [ ]:

client_rows = []
for run_key, run_output in experiment_results.items():
    counts = np.zeros(NUM_CLIENTS, dtype=int)
    for selected_clients in run_output["selected_history"]:
        for cid in selected_clients: counts[cid] += 1
    for cid, count in enumerate(counts):
        client_rows.append({"run": run_key, "client_id": cid,
                            "participation_count": int(count),
                            "participation_rate": float(count / max(EXPERIMENT_ROUNDS, 1))})
client_participation_df = pd.DataFrame(client_rows)
client_participation_df.to_csv(f"{OUT_DIR}/client_participation.csv", index=False)
participation_summary = client_participation_df.groupby("run").agg(
    min_participations=("participation_count", "min"),
    max_participations=("participation_count", "max"),
    std_participations=("participation_count", "std"),
    starved_clients=("participation_count", lambda x: int((x == 0).sum()))
).reset_index()
display(participation_summary)


In [ ]:

MODEL_UPLOAD_DOWNLOAD_MB = 2.0 * MODEL_SIZE_MB
communication_rows = []
for run_key, run_output in experiment_results.items():
    for round_no, selected_clients in enumerate(run_output["selected_history"], start=1):
        communication_rows.append({"run": run_key, "round": round_no,
                                   "selected_clients": len(selected_clients),
                                   "modeled_communication_mb": len(selected_clients) * MODEL_UPLOAD_DOWNLOAD_MB})
communication_volume_df = pd.DataFrame(communication_rows)
communication_volume_df.to_csv(f"{OUT_DIR}/communication_volume.csv", index=False)

resource_group_rows = []
for run_key, run_output in experiment_results.items():
    seed_value = int(re.search(r"seed(\d+)$", run_key).group(1)) if re.search(r"seed(\d+)$", run_key) else SEED
    res = initialize_client_resources(NUM_CLIENTS, seed_value)
    cpu_med = np.median([res[i]["cpu"] for i in range(NUM_CLIENTS)])
    bw_med = np.median([res[i]["bandwidth"] for i in range(NUM_CLIENTS)])
    counts = np.zeros(NUM_CLIENTS, dtype=int)
    for selected_clients in run_output["selected_history"]:
        for cid in selected_clients: counts[cid] += 1
    for cid in range(NUM_CLIENTS):
        group = "high-resource" if res[cid]["cpu"] >= cpu_med and res[cid]["bandwidth"] >= bw_med else "other-resource"
        resource_group_rows.append({"run": run_key, "client_id": cid, "resource_group": group,
                                    "participation_count": int(counts[cid])})
resource_fairness_df = pd.DataFrame(resource_group_rows)
resource_fairness_summary = resource_fairness_df.groupby(["run", "resource_group"]).participation_count.mean().reset_index()
resource_fairness_summary.to_csv(f"{OUT_DIR}/resource_group_participation.csv", index=False)
display(communication_volume_df.groupby("run").modeled_communication_mb.sum().reset_index())
display(resource_fairness_summary)


In [ ]:

REWARD_WEIGHT_SCENARIOS = {
    "accuracy_emphasis": {"accuracy": 0.60, "utility": 0.10, "time": 0.10, "communication": 0.05, "energy": 0.05, "fairness": 0.10},
    "balanced": {"accuracy": 0.45, "utility": 0.15, "time": 0.15, "communication": 0.10, "energy": 0.10, "fairness": 0.05},
    "efficiency_emphasis": {"accuracy": 0.35, "utility": 0.10, "time": 0.20, "communication": 0.10, "energy": 0.15, "fairness": 0.10},
}
for scenario, weights in REWARD_WEIGHT_SCENARIOS.items():
    assert abs(sum(weights.values()) - 1.0) < 1e-8, f"Weights must sum to 1: {scenario}"
print("Reward sensitivity scenarios defined:", list(REWARD_WEIGHT_SCENARIOS))
print("To execute a scenario, temporarily assign its values to W_GLOBAL_ACCURACY, W_DATA_UTILITY, W_TIME, W_COMMUNICATION, W_ENERGY, W_FAIRNESS, then rerun all methods with the same seeds.")


In [ ]:

PROPOSED_METHOD = "PPO_DQN"
ABLATION_FIXED_CLIENT_COUNT = 10
ABLATION_FIXED_LOCAL_EPOCHS = 2

ABLATION_CONFIGS = {
    "FULL_PROPOSED": {
        "method": PROPOSED_METHOD,
        "mode": None,
        "client_count": "PPO",
        "client_selection": "DQN",
        "local_epochs": "DQN",
    },
    "ABLATION_1_FIXED_COUNT": {
        "method": PROPOSED_METHOD,
        "mode": "FIXED_COUNT",
        "client_count": f"Fixed ({ABLATION_FIXED_CLIENT_COUNT})",
        "client_selection": "DQN",
        "local_epochs": "DQN",
    },
    "ABLATION_2_RANDOM_SELECTION": {
        "method": PROPOSED_METHOD,
        "mode": "RANDOM_SELECTION",
        "client_count": "PPO",
        "client_selection": "Random",
        "local_epochs": "DQN",
    },
    "ABLATION_3_FIXED_EPOCHS": {
        "method": PROPOSED_METHOD,
        "mode": "FIXED_EPOCHS",
        "client_count": "PPO",
        "client_selection": "DQN",
        "local_epochs": f"Fixed ({ABLATION_FIXED_LOCAL_EPOCHS})",
    },
}

RUN_ABLATIONS = True
ABLATION_SEEDS = RUN_SEEDS
ablation_results = {}

if RUN_ABLATIONS:
    for seed in ABLATION_SEEDS:
        for experiment_name, config in ABLATION_CONFIGS.items():
            key = f"{experiment_name}_seed{seed}"
            print(f"\nRunning {key}: {config}")
            ablation_results[key] = run_three_decision_experiment(
                method=PROPOSED_METHOD,
                num_rounds=EXPERIMENT_ROUNDS,
                seed=seed,
                verbose=True,
                ablation_mode=config["mode"],
            )

    ablation_summary = pd.DataFrame([
        {
            **result["summary"],
            "run": run_name,
            "experiment": run_name.rsplit("_seed", 1)[0],
            "seed": int(run_name.rsplit("_seed", 1)[1]),
            "client_count_policy": ABLATION_CONFIGS[run_name.rsplit("_seed", 1)[0]]["client_count"],
            "client_selection_policy": ABLATION_CONFIGS[run_name.rsplit("_seed", 1)[0]]["client_selection"],
            "local_epoch_policy": ABLATION_CONFIGS[run_name.rsplit("_seed", 1)[0]]["local_epochs"],
        }
        for run_name, result in ablation_results.items()
    ])

    os.makedirs(OUT_DIR, exist_ok=True)
    ablation_summary.to_csv(f"{OUT_DIR}/ablation_summary.csv", index=False)
    display(ablation_summary)
    print(f"Saved ablation summary to: {OUT_DIR}/ablation_summary.csv")
else:
    print("Ablation configurations are ready. Set RUN_ABLATIONS=True to execute them.")
    display(pd.DataFrame([
        {
            "Experiment": name,
            "Client count": cfg["client_count"],
            "Client selection": cfg["client_selection"],
            "Local epochs": cfg["local_epochs"],
        }
        for name, cfg in ABLATION_CONFIGS.items()
    ]))


In [ ]:

ROBUSTNESS_SCENARIOS = {
    "normal": {"availability_multiplier": 1.00, "extra_dropout_probability": 0.00, "post_selection_dropout_probability": 0.00},
    "moderate_dropout": {"availability_multiplier": 0.75, "extra_dropout_probability": 0.10, "post_selection_dropout_probability": 0.10},
    "severe_dropout": {"availability_multiplier": 0.50, "extra_dropout_probability": 0.25, "post_selection_dropout_probability": 0.25},
}
RUN_ROBUSTNESS = True
ROBUSTNESS_METHODS = ["DQN3", "DDQN3", "PPO_DQN"]
ROBUSTNESS_SEEDS = RUN_SEEDS

if RUN_ROBUSTNESS:
    _old_avail, _old_extra, _old_post = AVAILABILITY_MULTIPLIER, EXTRA_CLIENT_DROPOUT_PROB, POST_SELECTION_DROPOUT_PROB
    robustness_rows=[]
    try:
        for scenario, params in ROBUSTNESS_SCENARIOS.items():
            AVAILABILITY_MULTIPLIER = params["availability_multiplier"]
            EXTRA_CLIENT_DROPOUT_PROB = params["extra_dropout_probability"]
            POST_SELECTION_DROPOUT_PROB = params["post_selection_dropout_probability"]
            for seed in ROBUSTNESS_SEEDS:
                for method_name in ROBUSTNESS_METHODS:
                    out = run_three_decision_experiment(method_name, EXPERIMENT_ROUNDS, seed, verbose=True)
                    row=dict(out["summary"]); row.update({"scenario":scenario,"seed":seed})
                    robustness_rows.append(row)
        robustness_df=pd.DataFrame(robustness_rows)
        robustness_df.to_csv(f"{OUT_DIR}/robustness_sensitivity_summary.csv",index=False)
        display(robustness_df)
    finally:
        AVAILABILITY_MULTIPLIER, EXTRA_CLIENT_DROPOUT_PROB, POST_SELECTION_DROPOUT_PROB = _old_avail, _old_extra, _old_post
else:
    print("Set RUN_ROBUSTNESS=True to run the configured dropout/availability sensitivity experiments.")


In [ ]:

def per_class_test_accuracy(state):
    model=CIFAR10CNN().to(DEVICE); load_model_state(model,state); model.eval()
    correct=np.zeros(NUM_CLASSES,dtype=np.int64); total=np.zeros(NUM_CLASSES,dtype=np.int64)
    with torch.no_grad():
        for images, labels in test_loader:
            images=images.to(DEVICE); labels=labels.to(DEVICE)
            predictions=model(images).argmax(1)
            for cls in range(NUM_CLASSES):
                mask=(labels==cls); total[cls]+=mask.sum().item()
                correct[cls]+=((predictions==labels)&mask).sum().item()
    return 100.0*correct/np.maximum(total,1)

class_rows=[]
for run_key, run_output in experiment_results.items():
    class_acc=per_class_test_accuracy(run_output["final_model_state"])
    for cls, acc in enumerate(class_acc):
        class_rows.append({"run":run_key,"class_id":cls,"class_name":classes[cls],"test_accuracy":float(acc)})
per_class_df=pd.DataFrame(class_rows)
per_class_df.to_csv(f"{OUT_DIR}/per_class_test_accuracy.csv",index=False)
display(per_class_df.pivot(index="class_name",columns="run",values="test_accuracy"))

tradeoff_rows=[]
for run_key, run_output in experiment_results.items():
    tradeoff_rows.append({"run":run_key,
                          "final_test_accuracy":run_output["summary"].get("final_test_accuracy",np.nan),
                          "mean_jain_fairness":run_output["summary"].get("mean_fairness",np.nan),
                          "starved_clients":int((client_participation_df[client_participation_df.run==run_key].participation_count==0).sum())})
accuracy_fairness_tradeoff_df=pd.DataFrame(tradeoff_rows)
accuracy_fairness_tradeoff_df.to_csv(f"{OUT_DIR}/accuracy_fairness_tradeoff.csv",index=False)
display(accuracy_fairness_tradeoff_df)
